# Dataset agreement and validation, 2013–2025

This notebook preserves the analyses from both archived `00_` notebooks. The manuscript **2×2 candidate figure** and **2×3 six-prompt agreement figure** have their own display cells, alongside every underlying summary table.

The publication window is **1 January 2013–31 December 2025 inclusive**. Original screenshots may contain 2026 papers and older metrics; this notebook calculates current results from the dated inputs. It never substitutes a historical picture or a three-model deployment result for missing six-prompt validation data.

| Figure | Content | Export prefix |
|---|---|---|
| Original manuscript layout, 2×2 | Annual model positives; consensus/rest counts; keyword profiles; semantic map and silhouette | `00_06_figure_consensus_validation` |
| Six-prompt agreement, 2×3 | Classifier agreement for each original prompt strategy | `00_04_figure_validation_agreement` |
| Validation performance, 2×2 | Accuracy, precision, recall and F1 | `00_03_figure_validation_performance` |
| Additional candidate diagnostics, 2×3 | Prediction/parsing rates, votes, pairwise agreement and annual trends | `00_01_figure_candidate_agreement` |
| Text profiles | Keyword comparison and 25 TF-IDF terms per direction | `00_02_figure_candidate_text` |
| Semantic diagnostics, 1×2 | Identical projection coloured by consensus group and publication year | `00_05_figure_semantic_diagnostics` |
| Consensus categories | All distinct consensus outcomes, including unparsed labels | `00_07_figure_consensus_groups` |

Figures use Helvetica, shared colours, bold uppercase left-aligned titles and 500-DPI PNG/PDF exports. Each has a matching caption. A missing panel is explicitly labelled incomplete; missing validation files are listed at the validation section rather than hiding the omitted figures.


In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()


## 1. Inputs and run options

Part A automatically reads `data/analysis/dataset/matched_ukb_full_final_2013_2025_three_model_labels.csv`. `UKB_COMBINED_LABELS_CSV` can override it with another full candidate-level CSV. Showcase+ and TRUE-only exports cannot replace the candidate pool.

**Semantic analysis is enabled**, as in the original notebook. Matching coordinates and metrics are reused first. Otherwise a balanced sample of up to 3,000 papers per group is embedded using MiniLM; this never reruns Qwen, Llama or Mistral. `SEMANTIC_LOCAL_ONLY=True` uses cached model weights without downloads. If the encoder cannot load, the original explicitly labelled TF-IDF/SVD fallback remains available. Set `RUN_SEMANTIC_ANALYSIS=False` to reuse a saved semantic result without computing a new one.

Part B needs the six original `predictions_p*.csv` validation files under `VALIDATION_OUTPUT_DIR`. They are a different dataset from the three-model candidate labels. Existing predictions are reused. New validation inference remains disabled; enabling it requires the positive/negative labelled files and explicit evaluation sample sizes. Partial or incompatible caches fail without a silent refit.


In [ ]:
RUN_AGREEMENT = True
RUN_VALIDATION = True
RUN_SEMANTIC_ANALYSIS = os.environ.get("UKB_RUN_SEMANTIC_ANALYSIS", "1") != "0"
SEMANTIC_LOCAL_ONLY = True
RUN_VALIDATION_INFERENCE = os.environ.get("UKB_RUN_VALIDATION_INFERENCE", "0") == "1"

COMBINED_LABELS_CSV = os.environ.get("UKB_COMBINED_LABELS_CSV", "").strip() or None
VALIDATION_POSITIVE_CSV = os.environ.get("UKB_VALIDATION_POSITIVE_CSV", "").strip() or None
VALIDATION_NEGATIVE_CSV = os.environ.get("UKB_VALIDATION_NEGATIVE_CSV", "").strip() or None
VALIDATION_N_POS = os.environ.get("UKB_VALIDATION_N_POS", "").strip() or None
VALIDATION_N_NEG = os.environ.get("UKB_VALIDATION_N_NEG", "").strip() or None
VALIDATION_OUTPUT_DIR = Path(os.environ.get("UKB_VALIDATION_OUTPUT_DIR", "").strip() or P.OUTPUT / "validation").expanduser()
if not VALIDATION_OUTPUT_DIR.is_absolute():
    VALIDATION_OUTPUT_DIR = P.ROOT / VALIDATION_OUTPUT_DIR

TABLE_DIR = P.TABLE_DATA_ANALYSIS / "00_dataset"
FIGURE_DIR = P.FIG_DATA_ANALYSIS / "00_dataset"
MAX_TFIDF_PER_GROUP = 20_000
MAX_SEMANTIC_PER_GROUP = 3_000
SEED = 42


In [ ]:
import pandas as pd
from IPython.display import Image, display
from utils.shared_style import load_style
from utils import data_analysis_00_agreement as A
from utils import data_analysis_00_validation as V

STYLE = load_style("00_dataset")
section_results = []

def run_section(name, enabled, function, **kwargs):
    if not enabled:
        result = {"section": name, "status": "SKIP", "detail": "Disabled in notebook configuration."}
        print(f"[SKIP] {name}: disabled in configuration.")
    else:
        try:
            result = function(**kwargs)
            result.setdefault("section", name)
            result.setdefault("detail", result.get("reason", "Completed."))
        except Exception as error:
            detail = f"{type(error).__name__}: {str(error).splitlines()[0]}"
            print(f"[FAIL] {name}: {detail}")
            result = {"section": name, "status": "FAIL", "detail": detail}
    section_results.append(result)
    return result

def show_current_figure(result, stem):
    # Only show artifacts generated by this run, never an unrelated stale PNG.
    paths = result.get("figures", result.get("figure_files", []))
    matches = [Path(path) for path in paths
               if Path(path).suffix == ".png" and Path(path).stem in (stem, stem + "_incomplete")]
    if matches:
        display(Image(filename=str(matches[0]), width=1100))
        print(P.raw_path(matches[0]))
    else:
        print(f"[SKIP] {stem}: {result.get('detail', result.get('reason', 'Source data unavailable.'))}")

def show_agreement_table(name):
    table = agreement_result.get("table_frames", {}).get(name)
    if table is not None:
        display(table)
    else:
        print(f"[SKIP] {name}: agreement analysis unavailable.")


## 2. Compute candidate diagnostics and inspect the corpus

All deployed classifier labels are read from the saved CSV. This runs the original overview, model/consensus summaries, annual counts, explicit mentions, keyword profiles, TF-IDF contrasts and semantic diagnostic. Tables and figures are exported once; later cells display those exact results.

Tables: `output/tables/data_analysis/00_dataset/three_model_agreement/`.
Figures: `output/figures/data_analysis/00_dataset/three_model_agreement/`.


In [ ]:
agreement_result = run_section(
    "Three-model agreement", RUN_AGREEMENT, A.run_agreement,
    input_path=COMBINED_LABELS_CSV,
    output_dir=TABLE_DIR / "three_model_agreement",
    figure_dir=FIGURE_DIR / "three_model_agreement",
    run_semantic=RUN_SEMANTIC_ANALYSIS,
    max_tfidf_per_group=MAX_TFIDF_PER_GROUP,
    max_semantic_per_group=MAX_SEMANTIC_PER_GROUP,
    seed=SEED, show_tables=False, show_figures=False,
    semantic_local_only=SEMANTIC_LOCAL_ONLY,
)


In [ ]:
show_agreement_table("overview")
show_agreement_table("model_summary")


## 3. Restored manuscript figure: consensus diagnostics (2×2)

**A**, Annual TRUE predictions from Qwen, Llama and Mistral. **B**, Annual unanimous-TRUE versus remaining candidate counts (log scale). **C**, Keyword profiles in the two groups. **D**, The sentence-embedding projection coloured by consensus, with its recomputed silhouette index.

The same four analyses shown in the original manuscript panel are reunited here. The map describes separation of model-defined groups; it is not independent validation accuracy. If semantic data are unavailable, only an explicitly named `_incomplete` version is exported.


In [ ]:
show_current_figure(agreement_result, "00_06_figure_consensus_validation")
if "semantic_metrics" in agreement_result.get("table_frames", {}):
    show_agreement_table("semantic_metrics")


## 4. Model agreement, consensus categories and annual trends

The original vote, consensus-group and vote-signature tables distinguish unanimous FALSE, disagreements and unparsed responses. TRUE-vote counts alone do not distinguish those outcomes, so the consensus-group bar chart is also retained. Pairwise agreement uses only labels parsed by both models; its CSV supplies the corresponding denominator.


In [ ]:
show_current_figure(agreement_result, "00_01_figure_candidate_agreement")
show_agreement_table("pairwise_agreement")
show_agreement_table("yearly")


In [ ]:
show_current_figure(agreement_result, "00_07_figure_consensus_groups")
show_agreement_table("vote_distribution")
show_agreement_table("group_distribution")
show_agreement_table("signature_distribution")


## 5. Explicit mentions, keyword profiles and TF-IDF contrasts

The original explicit-mention table and full keyword counts are visible below. The text figure again shows up to **25 terms per direction**. The CSV retains all term scores; the tables below show the same leading terms with both group means and their difference.


In [ ]:
show_current_figure(agreement_result, "00_02_figure_candidate_text")
show_agreement_table("explicit_summary")
show_agreement_table("category_summary")
terms = agreement_result.get("table_frames", {}).get("tfidf_terms")
if terms is not None:
    difference = "difference_TRUE_minus_rest"
    display(terms.loc[terms[difference].gt(0)].nlargest(25, difference))
    display(terms.loc[terms[difference].lt(0)].nsmallest(25, difference))


## 6. Semantic diagnostic by group and publication year

Both original semantic views are preserved. They use the same sampled papers, coordinates and axes, with group labels on the left and publication year on the right. Sample identities, texts and years must match before cached coordinates can be reused. Metrics refer to the full embedding space; the figure is a two-dimensional projection.


In [ ]:
show_current_figure(agreement_result, "00_05_figure_semantic_diagnostics")


## 7. Validation inputs and saved model results

The original six-prompt validation analysis is retained below. If its files are absent, the notebook lists the exact inputs required and still completes the candidate analyses above. Supplying candidate labels does not supply these separate prompt-specific validation predictions.

**Encoder interpretation:** the original SciBERT and MiniLM baselines optimise their similarity thresholds on the evaluation labels. Their reported performance is in-sample calibration, not independent held-out validation.


In [ ]:
validation_availability = V.validation_cache_status(VALIDATION_OUTPUT_DIR)
display(pd.DataFrame(validation_availability["figure_status"]))
if validation_availability["missing_inputs"]:
    display(pd.DataFrame({"Missing validation input": [P.raw_path(path) for path in validation_availability["missing_inputs"]]}))


In [ ]:
validation_result = run_section(
    "Labelled validation", RUN_VALIDATION, V.run_validation,
    output_dir=VALIDATION_OUTPUT_DIR,
    positive_csv=VALIDATION_POSITIVE_CSV,
    negative_csv=VALIDATION_NEGATIVE_CSV,
    n_positive=VALIDATION_N_POS, n_negative=VALIDATION_N_NEG,
    run_inference=RUN_VALIDATION_INFERENCE,
    figure_dir=FIGURE_DIR / "validation",
    show_figures=False,
)


## 8. Original six-prompt pairwise-agreement figure (2×3)

**A–F**, Conservative, balanced, evidence-cue, context/no-shot, one-shot and five-shot prompts. Each matrix compares all available classifiers using papers with two parsed predictions. The original six-panel layout is retained, with shared colours, cell borders and paired-paper counts. Agreement measures consistency rather than accuracy against ground truth.


In [ ]:
show_current_figure(validation_result, "00_04_figure_validation_agreement")


## 9. Validation performance and model rankings (2×2)

**A–D**, Accuracy, precision, recall and F1 for each model and prompt. The full metrics table includes parse coverage and confusion-matrix counts. Rankings are shown separately; encoder rows remain explicitly labelled as in-sample results. Original runtime and throughput values are retained when present in saved results.


In [ ]:
show_current_figure(validation_result, "00_03_figure_validation_performance")
if validation_result.get("status") == "PASS":
    display(validation_result["summary"])
    display(validation_result["ranked"])


## 10. Output inventory and completion status

The inventory lists this run's actual tables and figures. Missing validation or semantic results stay visible as skipped/incomplete analyses; a successful notebook execution does not imply those results were available. Failures are raised after both analysis sections have been attempted.


In [ ]:
artifact_rows = []
for result in (agreement_result, validation_result):
    paths = [*result.get("tables", []), *result.get("figures", []), *result.get("files", [])]
    for path in dict.fromkeys(map(str, paths)):
        path = Path(path)
        if path.is_file():
            artifact_rows.append({"section": result["section"], "file": path.name,
                                  "path": P.raw_path(path), "size_kb": round(path.stat().st_size / 1024, 1)})
if artifact_rows:
    inventory = pd.DataFrame(artifact_rows)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    inventory.to_csv(TABLE_DIR / "00_dataset_artifact_inventory.csv", index=False)
    display(inventory)


In [ ]:
summary = pd.DataFrame([
    {"section": result["section"], "status": result["status"], "detail": result["detail"]}
    for result in section_results
])
if agreement_result.get("semantic_status") == "SKIP":
    summary.loc[len(summary)] = ["Semantic separation", "SKIP", "No matching semantic cache; encoding disabled."]
TABLE_DIR.mkdir(parents=True, exist_ok=True)
status_path = TABLE_DIR / "00_dataset_status.csv"
summary.to_csv(status_path, index=False)
display(summary)
counts = summary["status"].value_counts()
print(f"{counts.get('PASS', 0)} completed, {counts.get('SKIP', 0)} skipped, {counts.get('FAIL', 0)} failed.")
print("Status:", P.raw_path(status_path))
if summary["status"].eq("FAIL").any():
    raise RuntimeError("Dataset analysis failed: " + "; ".join(summary.loc[summary["status"].eq("FAIL"), "detail"]))
